In [2]:
#Cody's Info
#Steam API Key: D055A982E240438C0651DD3B290894B3 - ISteamuserstats
#Steam ID: 76561199121994793

#Matt's Info
#Steam API Key: 8718028FBB50D15D16CE98F13624C382 - ISteamuser
#Steam ID: 76561198244049512

from google.colab import drive
drive.mount('/content/drive')

# Set your output folder
output_dir = "/content/drive/MyDrive/Data Wrangling Project/GamingGuys/"

import os
os.makedirs(output_dir, exist_ok=True)
import os
print(f"Working directory: {os.getcwd()}")

import requests
import pandas as pd
import time

# Configuration — Matt & Cody only
users = [
    {"steam_id": "76561199121994793", "api_key": "D055A982E240438C0651DD3B290894B3", "label": "Cody"},
    {"steam_id": "76561198244049512", "api_key": "8718028FBB50D15D16CE98F13624C382", "label": "Matt"}
]

# DATA COLLECTION FUNCTIONS

def get_player_summaries(steam_ids, api_key):
    chunks = [steam_ids[i:i+100] for i in range(0, len(steam_ids), 100)]
    all_players = []
    for chunk in chunks:
        url = "http://api.steampowered.com/ISteamUser/GetPlayerSummaries/v2/"
        params = {"key": api_key, "steamids": ",".join(chunk)}
        response = requests.get(url, params=params).json()
        players = response["response"].get("players", [])
        all_players.extend(players)
    df = pd.DataFrame(all_players)
    for col in ["steamid", "personaname", "loccountrycode", "timecreated"]:
        if col not in df.columns:
            df[col] = None
    df = df[["steamid", "personaname", "loccountrycode", "timecreated"]].rename(columns={
        "personaname": "username",
        "loccountrycode": "country"
    })
    return df

def get_owned_games(steam_ids, api_key):
    all_dfs = []
    url = "http://api.steampowered.com/IPlayerService/GetOwnedGames/v1/"
    for sid in steam_ids:
        params = {"key": api_key, "steamid": sid, "include_appinfo": True, "include_played_free_games": True}
        response = requests.get(url, params=params).json()
        games = response["response"].get("games", [])
        if not games:
            print(f"No data for SteamID {sid} — profile may be private or empty.")
            continue
        temp = pd.DataFrame(games)
        temp["steamid"] = sid
        all_dfs.append(temp)
        time.sleep(0.5)
    if not all_dfs:
        return pd.DataFrame()
    df = pd.concat(all_dfs, ignore_index=True)
    df["playtime_hours"] = df["playtime_forever"] / 60
    df = df[df["playtime_hours"] > 0]
    df = df.drop(columns=["playtime_forever"])
    df = df[["steamid", "appid", "name", "playtime_hours"]]
    return df

def get_achievements(steam_ids, app_ids, api_key):
    all_dfs = []
    for sid in steam_ids:
        for appid in app_ids:
            url = "http://api.steampowered.com/ISteamUserStats/GetPlayerAchievements/v1/"
            params = {"key": api_key, "steamid": sid, "appid": appid}
            response = requests.get(url, params=params).json()
            stats = response.get("playerstats", {})
            if not stats.get("success") or "achievements" not in stats:
                continue
            achievements = pd.DataFrame(stats["achievements"])
            completed = achievements[achievements["achieved"] == 1].shape[0]
            total = achievements.shape[0]
            all_dfs.append({"steamid": sid, "appid": appid,
                           "achievements_completed": completed, "achievements_total": total})
            time.sleep(0.5)
    if not all_dfs:
        return pd.DataFrame(columns=["steamid", "appid", "achievements_completed", "achievements_total"])
    return pd.DataFrame(all_dfs)

def get_steamspy_data(app_ids):
    all_dfs = []
    for appid in app_ids:
        url = f"https://steamspy.com/api.php?request=appdetails&appid={appid}"
        response = requests.get(url).json()
        try:
            price_usd = int(response.get("price", 0)) / 100
        except (ValueError, TypeError):
            price_usd = None
        all_dfs.append({
            "appid": appid, "steamspy_name": response.get("name"),
            "genre": response.get("genre"), "tags": response.get("tags"),
            "owners": response.get("owners"), "positive_ratings": response.get("positive"),
            "negative_ratings": response.get("negative"), "price_usd": price_usd,
            "discount": response.get("discount"), "average_playtime": response.get("average_forever")
        })
        time.sleep(1)
    return pd.DataFrame(all_dfs)

# BUILD RAW MASTER DATAFRAME

def build_master_df(users):
    all_frames = []
    for user in users:
        sid, key, label = user["steam_id"], user["api_key"], user["label"]
        print(f"\nPulling data for {label}...")
        players_df = get_player_summaries([sid], key)
        games_df = get_owned_games([sid], key)
        if games_df.empty:
            print(f"  No game data for {label}, skipping.")
            continue
        app_ids = games_df["appid"].unique().tolist()
        achievements_df = get_achievements([sid], app_ids, key)
        steamspy_df = get_steamspy_data(app_ids)
        df = games_df.merge(players_df, on="steamid", how="left")
        df = df.merge(achievements_df, on=["steamid", "appid"], how="left")
        df = df.merge(steamspy_df, on="appid", how="left")
        df["label"] = label
        all_frames.append(df)
    all_frames = [f for f in all_frames if not f.empty]
    return pd.concat(all_frames, ignore_index=True)

master_df = build_master_df(users)

print(f"\n{'='*60}")
print(f"RAW DATA — NO WRANGLING YET")
print(f"{'='*60}")
print(f"Shape: {master_df.shape}")
print(f"Columns: {list(master_df.columns)}")
print(f"\nDtypes:\n{master_df.dtypes}")
print(f"\nMissing values:\n{master_df.isnull().sum()}")
print(f"\nFirst 3 rows:")
print(master_df.head(3).to_string())

master_df.to_csv(f"{output_dir}steam_v1_raw.csv", index=False)
print("\nExported: steam_v1_raw.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content

Pulling data for Cody...

Pulling data for Matt...

RAW DATA — NO WRANGLING YET
Shape: (130, 19)
Columns: ['steamid', 'appid', 'name', 'playtime_hours', 'username', 'country', 'timecreated', 'achievements_completed', 'achievements_total', 'steamspy_name', 'genre', 'tags', 'owners', 'positive_ratings', 'negative_ratings', 'price_usd', 'discount', 'average_playtime', 'label']

Dtypes:
steamid                    object
appid                       int64
name                       object
playtime_hours            float64
username                   object
country                    object
timecreated                 int64
achievements_completed    float64
achievements_total        float64
steamspy_name              object
genre                      object
tags                       object
owners                     object
positive_ratin

/tmp/ipykernel_18243/3427278140.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_frames, ignore_index=True)
